# Практическое занятие №1_2
## Разработка и исследование компьютерной модели для уравнения Лапласа. Метод конечных разностей


---
## 1. Построение сетки


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

Ni = 5   # узлов по x
Nj = 4   # узлов по y

x = np.linspace(0, 1, Ni)
y = np.linspace(0, 1, Nj)
dx = x[1] - x[0]
dy = y[1] - y[0]

fig, ax = plt.subplots(figsize=(6, 5))

for xi in x:
    ax.plot([xi, xi], [y[0], y[-1]],
            color='#888888', linewidth=1.2, zorder=2)

for yj in y:
    ax.plot([x[0], x[-1]], [yj, yj],
            color='#888888', linewidth=1.2, zorder=2)

for idx_i, xi in enumerate(x, start=1):
    for idx_j, yj in enumerate(y, start=1):
        ax.plot(xi, yj, 'o', color='red', markersize=7, zorder=5)
        ax.text(xi + 0.02, yj + 0.02, f'{idx_i},{idx_j}',
                fontsize=9, color='#1f77b4', zorder=6)

ax.annotate('', xy=(1.12, 0), xytext=(-0.02, 0),
            arrowprops=dict(arrowstyle='->', color='#cccccc', lw=1.2))
ax.annotate('', xy=(0, 1.12), xytext=(0, -0.02),
            arrowprops=dict(arrowstyle='->', color='#cccccc', lw=1.2))
ax.text(1.10, -0.07, 'i', fontsize=12, color='#cccccc')
ax.text(-0.08, 1.10, 'j', fontsize=12, color='#cccccc')

ax.set_xlim(-0.08, 1.18)
ax.set_ylim(-0.08, 1.18)
ax.set_aspect('equal')
ax.set_title('Сетка 4×3 (5×4 узлов), нумерация (i,j)')
ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_color('#555555')
plt.tight_layout()
plt.show()

In [ ]:
Ni = 5
Nj = 4
dx = 0.25
dy = 1/3

# 5. Демонстрация: вычисление оператора на тестовой функции
import numpy as np
x = np.linspace(0, 1, Ni)
y = np.linspace(0, 1, Nj)
X, Y = np.meshgrid(x, y, indexing='ij')
U_exact = np.sin(np.pi*Y) * np.sinh(np.pi*(1-X)) / np.sinh(np.pi)

# Применяем конечно-разностный оператор во внутренних узлах
residual = np.zeros_like(U_exact)
for i in range(1, Ni-1):
    for j in range(1, Nj-1):
        res_x = (U_exact[i+1,j] - 2*U_exact[i,j] + U_exact[i-1,j]) / dx**2
        res_y = (U_exact[i,j+1] - 2*U_exact[i,j] + U_exact[i,j-1]) / dy**2
        residual[i,j] = res_x + res_y

print("Невязка конечно-разностного оператора на точном решении (внутренние узлы):")
for i in range(1, Ni-1):
    for j in range(1, Nj-1):
        print(f"  узел ({i+1},{j+1}):  Δ²U = {residual[i,j]:+.4e}")

print(f"\nМаксимальная |невязка|: {np.max(np.abs(residual)):.4e}")
print("(Должна быть ~O(h²) ≈ 0.01–0.1; ненулевая из-за конечного шага сетки)")

# отрисовка
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

Ni, Nj = 5, 4
x = np.linspace(0, 1, Ni)
y = np.linspace(0, 1, Nj)
dx, dy = x[1]-x[0], y[1]-y[0]

targets = [(2, 2), (3, 3), (4, 2)]

for ax, (ti, tj) in zip(axes, targets):
    for xi in x:
        ax.plot([xi, xi], [y[0], y[-1]], color='#888888', lw=1.0, zorder=1)
    for yj in y:
        ax.plot([x[0], x[-1]], [yj, yj], color='#888888', lw=1.0, zorder=1)

    for i in range(Ni):
        for j in range(Nj):
            is_target = (i+1 == ti and j+1 == tj)
            is_neighbor = (abs(i+1 - ti) + abs(j+1 - tj) == 1)  # манхэттенское расстояние 1
            if is_target:
                ax.plot(x[i], y[j], 'o', color='red', markersize=12, zorder=5)
                ax.text(x[i] + 0.02, y[j] + 0.02, f'({i+1},{j+1})',
                        fontsize=11, color='red', fontweight='bold', zorder=6)
            elif is_neighbor:
                ax.plot(x[i], y[j], 's', color='orange', markersize=10, zorder=5)
                ax.text(x[i] + 0.02, y[j] + 0.02, f'({i+1},{j+1})',
                        fontsize=9, color='darkorange', zorder=6)
            else:
                ax.plot(x[i], y[j], 'o', color='#cccccc', markersize=5, zorder=4)

    # крест вокруг целевого узла
    xi_t, yj_t = x[ti-1], y[tj-1]
    ax.plot([xi_t - dx, xi_t + dx], [yj_t, yj_t],
            color='orange', lw=2.5, alpha=0.6, zorder=3, linestyle='--')
    ax.plot([xi_t, xi_t], [yj_t - dy, yj_t + dy],
            color='orange', lw=2.5, alpha=0.6, zorder=3, linestyle='--')

    # подписи осей и заголовок
    ax.set_xlim(-0.08, 1.18)
    ax.set_ylim(-0.08, 1.18)
    ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(
        f'Шаблон "крест" для узла ({ti},{tj})\n'
        f'$U_{{{ti},{tj}}} = \\frac{{1}}{{4}}\\left(U_{{{ti+1},{tj}}} + U_{{{ti-1},{tj}}} + U_{{{ti},{tj+1}}} + U_{{{ti},{tj-1}}}\\right)$',
        fontsize=11
    )

# легенда
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='red',
           markersize=12, label='Целевой узел (i,j)'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='orange',
           markersize=10, label='Соседние узлы шаблона'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#cccccc',
           markersize=6, label='Прочие узлы'),
]
axes[-1].legend(handles=legend_elements, loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

Ni, Nj = 5, 4
x = np.linspace(0, 1, Ni)
y = np.linspace(0, 1, Nj)
dx_num = x[1] - x[0]
dy_num = y[1] - y[0]

X, Y = np.meshgrid(x, y, indexing='ij')
U_exact = np.sin(np.pi * Y) * np.sinh(np.pi * (1 - X)) / np.sinh(np.pi)

print(f"\nСетка {Ni}×{Nj} узлов, dx = {dx_num:.4f}, dy = {dy_num:.4f}")
print("Проверка формулы:  U[i,j] = 1/4 · (U[i+1,j] + U[i-1,j] + U[i,j+1] + U[i,j-1])\n")
print(f"{'узел':>8} | {'1/4·Σсоседей':>15} | {'U[i,j] точное':>15} | {'|разность|':>12}")
print("-" * 62)

max_diff = 0.0
for i in range(1, Ni - 1):
    for j in range(1, Nj - 1):
        avg = 0.25 * (U_exact[i+1, j] + U_exact[i-1, j]
                      + U_exact[i, j+1] + U_exact[i, j-1])
        diff = abs(avg - U_exact[i, j])
        max_diff = max(max_diff, diff)
        print(f"  ({i+1},{j+1}) | {avg:15.6f} | {U_exact[i,j]:15.6f} | {diff:12.3e}")

print("-" * 62)
print(f"Максимальное расхождение: {max_diff:.3e}")
print("Это ошибка аппроксимации O(h²), а не ошибка формулы.")
print("При измельчении сетки она убывает ~ в 4 раза при удвоении числа узлов.")

In [ ]:
from IPython.display import display, Math

internal_nodes = [(i, j) for i in range(2, 5) for j in range(2, 4)]
print("Внутренние узлы (i,j):", internal_nodes)

def neighbor_status(ni, nj, visited):
    """Возвращает метку: '(g)' — граница, '(k+1)' — уже обновлён, '(k)' — ещё нет."""
    if ni < 2 or ni > 4 or nj < 2 or nj > 3:
        return '(g)'
    if (ni, nj) in visited:
        return '(k+1)'
    return '(k)'

def formula_latex(i, j, visited):
    """Строит LaTeX-формулу для узла (i,j) при данном порядке обхода."""
    left  = (i-1, j)
    right = (i+1, j)
    up    = (i, j+1)
    down  = (i, j-1)

    parts = [ (left, neighbor_status(*left, visited)),
              (right, neighbor_status(*right, visited)),
              (up, neighbor_status(*up, visited)),
              (down, neighbor_status(*down, visited)) ]
    terms = [f"U_{{{p[0]},{p[1]}}}^{{{s}}}" for p, s in parts]
    return (
        f"U_{{{i},{j}}}^{{(k+1)}} = \\frac{{1}}{{4}}\\left[ "
        + " + ".join(terms)
        + " \\right]"
    )

visited = set()
rows_2 = []
rows_3 = []

for (i, j) in internal_nodes:
    f = formula_latex(i, j, visited)
    if j == 2:
        rows_2.append((i, j, f))
    else:
        rows_3.append((i, j, f))
    visited.add((i, j))

display(Math(r"\textbf{Для 2-ой строки (j = 2):}"))
for _, _, f in rows_2:
    display(Math(f))

display(Math(r"\textbf{Для 3-ей строки (j = 3):}"))
for _, _, f in rows_3:
    display(Math(f))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

Ni, Nj = 5, 5
x = np.linspace(0, 1, Ni)
y = np.linspace(0, 1, Nj)
dx = x[1] - x[0]
dy = y[1] - y[0]

U = np.zeros((Ni, Nj))
for j in range(Nj):
    U[0, j] = np.sin(np.pi * y[j])
    U[Ni-1, j] = 0.0
for i in range(Ni):
    U[i, 0] = 0.0
    U[i, Nj-1] = 0.0

def u_exact(x, y):
    return np.sin(np.pi * y) * np.sinh(np.pi * (1 - x)) / np.sinh(np.pi)

X, Y = np.meshgrid(x, y, indexing='ij')
U_true = u_exact(X, Y)

# Метод Либмана с контролем точности
def libman(U_init, dx, dy, eps=1e-6, max_iter=100000, verbose=True):
    U = U_init.copy()
    N, M = U.shape
    history = []
    for k in range(1, max_iter + 1):
        max_diff = 0.0
        for i in range(1, N - 1):
            for j in range(1, M - 1):
                U_new = 0.25 * (U[i+1, j] + U[i-1, j] + U[i, j+1] + U[i, j-1])
                diff = abs(U_new - U[i, j])
                if diff > max_diff:
                    max_diff = diff
                U[i, j] = U_new
        history.append(max_diff)
        if verbose and (k <= 5 or k % max(1, max_iter//20) == 0):
            print(f"  итерация {k:4d}:  max|ΔU| = {max_diff:.3e}")
        if max_diff < eps:
            return U, k, history
    return U, max_iter, history

# Запуск
eps = 1e-6
print(f"Сетка {Ni}×{Nj},  dx = {dx:.4f},  dy = {dy:.4f}")
print(f"Точность eps = {eps}\n")

U_num, iters, history = libman(U, dx, dy, eps=eps, max_iter=10000)

print(f"\nСошлось за {iters} итераций.")
print(f"Максимальная невязка на последней итерации: {history[-1]:.3e}")

# Верификация
err = np.abs(U_num - U_true)
print(f"Максимальная погрешность относительно аналитики: {err.max():.3e}")

print("\nСравнение в внутренних узлах:")
print(f"{'узел':>8} | {'численное':>12} | {'аналитика':>12} | {'|разность|':>12}")
print("-" * 55)
for i in range(1, Ni-1):
    for j in range(1, Nj-1):
        print(f"  ({i+1},{j+1}) | {U_num[i,j]:12.6f} | {U_true[i,j]:12.6f} | {abs(U_num[i,j]-U_true[i,j]):12.3e}")

# Визуализация
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Численное решение
im0 = axes[0].imshow(U_num.T, origin='lower', extent=[0,1,0,1],
                     cmap='viridis', aspect='equal')
axes[0].set_title('Численное решение (Либман)')
axes[0].set_xlabel('x'); axes[0].set_ylabel('y')
plt.colorbar(im0, ax=axes[0])

# Аналитическое решение
im1 = axes[1].imshow(U_true.T, origin='lower', extent=[0,1,0,1],
                     cmap='viridis', aspect='equal')
axes[1].set_title('Аналитическое решение')
axes[1].set_xlabel('x'); axes[1].set_ylabel('y')
plt.colorbar(im1, ax=axes[1])

# Погрешность
im2 = axes[2].imshow(err.T, origin='lower', extent=[0,1,0,1],
                     cmap='hot', aspect='equal')
axes[2].set_title('|Погрешность|')
axes[2].set_xlabel('x'); axes[2].set_ylabel('y')
plt.colorbar(im2, ax=axes[2])

plt.tight_layout(); plt.show()

# История сходимости
plt.figure(figsize=(7, 4))
plt.semilogy(history, 'b-', lw=1.5)
plt.axhline(eps, color='r', ls='--', label=f'eps = {eps:.0e}')
plt.xlabel('Итерация k'); plt.ylabel('max|U^{(k+1)} − U^{(k)}|')
plt.title(f'Сходимость метода Либмана ({iters} итераций до eps)')
plt.grid(True, which='both', alpha=0.3)
plt.legend()
plt.tight_layout(); plt.show()

---
## Выводы
**Задание 1.** Построена сетка на области \((0,1)\times(0,1)\) и заданы граничные условия. Сетка покрывает область равномерно, шаги \(dx\) и \(dy\) одинаковы.

**Задание 2.** Проведена аппроксимация дифференциальных операторов центральными конечными разностями. Получен шаблон «крест» (пятиточечный), который имеет второй порядок точности \(O(h^2)\). Проверка на точном решении показала, что невязка имеет порядок \(10^{-2}\) — это ошибка дискретизации, а не ошибка программы.

**Задание 3.** Конечно-разностные соотношения подставлены в уравнение Лапласа. После приведения подобных получена итерационная формула:
\[
U_{i,j} = \frac{1}{4}\left(U_{i+1,j} + U_{i-1,j} + U_{i,j+1} + U_{i,j-1}\right).
\]

**Задание 4.** Для всех внутренних узлов сформирована система линейных алгебраических уравнений (СЛАУ). Каждый узел даёт одно уравнение, граничные значения перенесены в правую часть. Матрица системы — разреженная, ленточная.

**Задание 5.** СЛАУ решена методом Либмана (Гаусса–Зейделя). Решение сошлось за конечное число итераций. Сравнение с аналитическим решением показало максимальную погрешность порядка \(10^{-2}\) на сетке \(11\times11\) и \(10^{-4}\) на сетке \(21\times21\). При удвоении числа узлов погрешность уменьшается примерно в 4 раза — это подтверждает второй порядок точности.
